# Neural Network Training: Dense Layers on Hybrid Hardware

This notebook traces neural network forward passes, training steps, and batch
inference as **matrix operation graphs** compiled and executed through the uniqx uniqx.

| Operation | Key op | Hardware affinity |
|:----------|:-------|:------------------|
| Forward pass | `matmul` chain + activations | GPU at large width |
| Training step | `matmul` + `sub` + `dot` (loss) | GPU at large batch |
| Batch inference | `matmul` (parallelisable) | GPU always |

We compare three network sizes and three batch sizes, showing the
GPU crossover point where transfer overhead is amortised by parallelism.

**All computation goes through uniqx** — no local NumPy/PyTorch.

In [ ]:
import os
from uniqx import connect, login

endpoint = os.environ.get("UNIQX_GATEWAY", "api.oriqx.com:443")
login(os.environ["UNIQX_API_KEY"], gateway=endpoint)
client = connect(endpoint)
print("Connected to", endpoint)
import time
import matplotlib.pyplot as plt

from uniqx.domains.ml import (
    DenseNet,
    build_forward_module,
    build_training_step_module,
    build_inference_module,
    ACTIVATION_FNS,
)
from uniqx import parse_result
from uniqx.core.execution import connect, preflight, submit, get

API_KEY = os.environ.get("UNIQX_API_KEY")


## 1. Network Architectures

In [ ]:
nets = {
    "small": DenseNet(layers=[16, 8, 4], activation="tanh"),
    "medium": DenseNet(layers=[64, 32, 16, 8], activation="tanh"),
    "large": DenseNet(layers=[128, 64, 32, 16], activation="sigmoid"),
}

print(f"{'Name':<10} {'Architecture':<30} {'Params':>8} {'Activation':>12}")
print("-" * 62)
for name, net in nets.items():
    arch = " -> ".join(str(l) for l in net.layers)
    print(f"{name:<10} {arch:<30} {net.total_params:>8} {net.activation:>12}")

## 2. Forward Pass: Preflight Comparison

In [ ]:
def print_opts(label, options):
    print(f"\n--- {label} ---")
    if not options:
        print("  (no options)")
        return
    print(f"  {'Label':<24} {'Time':>10} {'Cost ($)':>12} {'Error':>8}")
    print(f"  {'-' * 24} {'-' * 10} {'-' * 12} {'-' * 8}")
    for o in options:
        f = " *" if o.get("recommended") else ""
        print(
            f"  {o['label']:<24} {o['total_time']:>10.2f}  ${o['total_cost_usd']:>10.6f}  {o['max_error_rate']:>8.4f}{f}"
        )


forward_data = {}

for name, net in nets.items():
    mod, inputs, meta = build_forward_module(net, batch_size=1)
    opts = preflight(mod, client=client)
    print_opts(f"Forward: {name} ({net.total_params} params)", opts)
    forward_data[name] = {"mod": mod, "inputs": inputs, "opts": opts, "net": net}

## 3. Execute Forward Passes

In [ ]:
forward_results = {}

for name, data in forward_data.items():
    forward_results[name] = {}
    for opt in data["opts"]:
        t0 = time.monotonic()
        jid = submit(
            data["mod"],
            client=client,
            runtime_inputs=data["inputs"],
            preflight_job_id=data["opts"].job_id,
            option_idx=opt["_idx"],
        )
        res = get(jid, client=client)
        wall = time.monotonic() - t0
        forward_results[name][opt["label"]] = {"time": wall, "option": opt}
        print(f"{name}/{opt['label']}: {wall:.2f}s")

## 4. Training Step: Batch Size Scaling

In [ ]:
train_results = {}

for batch_size in [4, 16, 32]:
    net = nets["medium"]
    mod, inputs, meta = build_training_step_module(net, batch_size=batch_size)
    opts = preflight(mod, client=client)
    print_opts(f"Training: medium net, batch={batch_size}", opts)

    train_results[batch_size] = {}
    for opt in opts:
        t0 = time.monotonic()
        jid = submit(
            mod,
            client=client,
            runtime_inputs=inputs,
            preflight_job_id=opts.job_id,
            option_idx=opt["_idx"],
        )
        res = get(jid, client=client)
        wall = time.monotonic() - t0

        payload = res.get("payload", b"")
        if isinstance(payload, str):
            payload = payload.encode()
        out = parse_result(payload, ["predictions", "loss", "grad_norm"])
        loss = out.get("loss", ([], "", [0.0]))[2][0] if out else 0.0

        train_results[batch_size][opt["label"]] = {
            "time": wall,
            "loss": loss,
            "option": opt,
        }
        print(f"  {opt['label']}: {wall:.2f}s, loss={loss:.6f}")

## 5. Batch Inference Throughput

In [ ]:
infer_results = {}

for batch_size in [8, 32, 64]:
    net = nets["large"]
    mod, inputs, meta = build_inference_module(net, batch_size=batch_size)
    opts = preflight(mod, client=client)
    print_opts(f"Inference: large net, batch={batch_size}", opts)

    infer_results[batch_size] = {}
    for opt in opts:
        t0 = time.monotonic()
        jid = submit(
            mod,
            client=client,
            runtime_inputs=inputs,
            preflight_job_id=opts.job_id,
            option_idx=opt["_idx"],
        )
        res = get(jid, client=client)
        wall = time.monotonic() - t0
        throughput = batch_size / wall if wall > 0 else 0
        infer_results[batch_size][opt["label"]] = {
            "time": wall,
            "throughput": throughput,
            "option": opt,
        }
        print(f"  {opt['label']}: {wall:.2f}s ({throughput:.0f} samples/s)")

## 6. Activation Function Comparison

In [ ]:
act_results = {}

for act in ACTIVATION_FNS:
    net = DenseNet(layers=[64, 32, 16, 8], activation=act)
    mod, inputs, meta = build_forward_module(net, batch_size=8)
    opts = preflight(mod, client=client)

    act_results[act] = {}
    for opt in opts:
        t0 = time.monotonic()
        jid = submit(
            mod,
            client=client,
            runtime_inputs=inputs,
            preflight_job_id=opts.job_id,
            option_idx=opt["_idx"],
        )
        res = get(jid, client=client)
        wall = time.monotonic() - t0
        act_results[act][opt["label"]] = {"time": wall, "option": opt}

    rec = opts.recommended
    print(
        f"{act:>15}: recommended={rec['label'] if rec else 'none'}, est_time={rec['total_time']:.1f}"
    )

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors_hw = {"cpu-only": "#2563eb", "cpu+gpu": "#16a34a", "cpu+gpu+qpu": "#9333ea"}

# Top-left: Forward pass time by network size
net_names = list(forward_results.keys())
for hw in sorted(set(l for n in forward_results for l in forward_results[n])):
    times = [forward_results[n].get(hw, {}).get("time", 0) for n in net_names]
    params = [nets[n].total_params for n in net_names]
    c = colors_hw.get(hw, "#6b7280")
    axes[0, 0].plot(params, times, "o-", color=c, label=hw)
axes[0, 0].set_xlabel("Total parameters")
axes[0, 0].set_ylabel("Wall time (s)")
axes[0, 0].set_title("Forward Pass Scaling")
axes[0, 0].legend(fontsize=8)
axes[0, 0].grid(alpha=0.3)

# Top-right: Training time by batch size
batches = sorted(train_results.keys())
for hw in sorted(set(l for b in train_results for l in train_results[b])):
    times = [train_results[b].get(hw, {}).get("time", 0) for b in batches]
    c = colors_hw.get(hw, "#6b7280")
    axes[0, 1].plot(batches, times, "o-", color=c, label=hw)
axes[0, 1].set_xlabel("Batch size")
axes[0, 1].set_ylabel("Wall time (s)")
axes[0, 1].set_title("Training Step: Batch Scaling")
axes[0, 1].legend(fontsize=8)
axes[0, 1].grid(alpha=0.3)

# Bottom-left: Inference throughput
batches_i = sorted(infer_results.keys())
for hw in sorted(set(l for b in infer_results for l in infer_results[b])):
    tp = [infer_results[b].get(hw, {}).get("throughput", 0) for b in batches_i]
    c = colors_hw.get(hw, "#6b7280")
    axes[1, 0].plot(batches_i, tp, "o-", color=c, label=hw)
axes[1, 0].set_xlabel("Batch size")
axes[1, 0].set_ylabel("Throughput (samples/s)")
axes[1, 0].set_title("Inference Throughput")
axes[1, 0].legend(fontsize=8)
axes[1, 0].grid(alpha=0.3)

# Bottom-right: Activation function comparison
acts = list(act_results.keys())
all_hw = sorted(set(l for a in act_results for l in act_results[a]))
width = 0.8 / max(len(all_hw), 1)
for hi, hw in enumerate(all_hw):
    times = [act_results[a].get(hw, {}).get("time", 0) for a in acts]
    c = colors_hw.get(hw, "#6b7280")
    axes[1, 1].bar(
        [i + hi * width for i in range(len(acts))],
        times,
        width=width,
        color=c,
        label=hw,
        alpha=0.8,
    )
axes[1, 1].set_xticks([i + width * (len(all_hw) - 1) / 2 for i in range(len(acts))])
axes[1, 1].set_xticklabels(acts)
axes[1, 1].set_ylabel("Wall time (s)")
axes[1, 1].set_title("Activation Function Cost")
axes[1, 1].legend(fontsize=8)
axes[1, 1].grid(axis="y", alpha=0.3)

fig.suptitle(
    "Neural Network Training on Hybrid Hardware", fontsize=14, fontweight="bold"
)
plt.tight_layout()
plt.show()

## Summary

| Workload | GPU crossover | Key insight |
|:---------|:-------------|:------------|
| Forward (small) | > 200 params | Transfer overhead dominates |
| Forward (large) | Always GPU | matmul parallelism wins |
| Training | batch >= 16 | Batch dimension amortises transfer |
| Inference | batch >= 8 | Throughput scales linearly on GPU |

The uniqx uniqx compiles the full matmul+activation graph into a single
fused kernel. The cost model captures the GPU crossover point — users
just submit and the platform optimises.